In [0]:
dbutils.widgets.removeAll()

In [0]:
import os
import sys
import shutil
from pathlib import Path
import json
import pandas as pd

In [0]:
# Force local file system synchronization
os.sync()

# Absolute workspace configuration
ROOT_DIR = Path("/Workspace/Repos/logi@openhealthagents.org/claimspan/ClaimsProcessing")
sys.path.append(str(ROOT_DIR))

In [0]:
%run ./caregap_analyzer

In [0]:
from Shared.EDIProcessing import EDIProcessor, CSVConverter
from FactGapsInCare.EDIProcessing.mapper import Mapper as ClaimsMapper

In [0]:
def move_file(src_path: Path, target_dir: Path) -> Path:
    """Moves a file to a target directory cleanly, ensuring the directory exists."""
    target_dir.mkdir(parents=True, exist_ok=True)
    target_path = target_dir / src_path.name
    shutil.move(str(src_path), str(target_path))
    return target_path

In [0]:
def normalize_claims_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """Standardizes column names for 837 claims data."""
    if df.empty:
        return df

    df.columns = df.columns.str.strip().str.lower()
    if "patient_dob" in df.columns and "dob" not in df.columns:
        df["dob"] = df["patient_dob"]
    if "patient_id" in df.columns and "member_id" not in df.columns:
        df["member_id"] = df["patient_id"]

    return df

In [0]:
def process_edi_claims_file(file_path: Path, base_source_dir: Path) -> tuple:
    """Parses 837 claims file, maps records, writes temp CSV, and returns Pandas DataFrame."""
    active_file_path = file_path
    try:
        if not active_file_path.exists():
            raise FileNotFoundError(f"Input file missing: {active_file_path}")
        
        # Move raw file to inprogress
        active_file_path = move_file(active_file_path, base_source_dir / "inprogress")

        # Parse EDI JSON & Map Claims
        structured_json = EDIProcessor().parse(str(active_file_path))
        mapped_records = ClaimsMapper().map_claims(structured_json)
        layout_id = "837"

        # Temp CSV File Generation
        target_csv_name = f"{active_file_path.stem}.csv"
        target_csv_path = ROOT_DIR / "temp" / layout_id / target_csv_name
        target_csv_path.parent.mkdir(parents=True, exist_ok=True)
        
        try:
            CSVConverter().converter(mapped_records, str(target_csv_path))
        except Exception as csv_error:
            print(f"Warning: CSV conversion failed: {csv_error}")
        
        if isinstance(mapped_records, list):
            df = pd.DataFrame(mapped_records)
        elif isinstance(mapped_records, dict):
            df = pd.DataFrame([mapped_records])
        elif target_csv_path.exists():
            df = pd.read_csv(target_csv_path)
        else:
            raise ValueError(f"Cannot convert mapped_records to DataFrame. Type: {type(mapped_records)}")

        print(f"[837] Processed {active_file_path.name} -> CSV created at {target_csv_path}")
        return df, active_file_path

    except Exception as e:
        print(f"Failed processing 837 file {active_file_path.name}: {e}")
        if active_file_path.exists():
            move_file(active_file_path, base_source_dir / "failed")
        raise


In [0]:
def finalize_file_tracking(file_path: Path, base_source_dir: Path, success: bool):
    """Moves the raw EDI file based on lifecycle completion status."""
    try:
        if success:
            print(f"--> Archiving raw file to processed: {file_path.name}")
            move_file(file_path, base_source_dir / "processed")
        else:
            print(f"--> Moving raw file to failed: {file_path.name}")
            move_file(file_path, base_source_dir / "failed")
    except Exception as e:
        print(f"Failed to update tracking directory state: {e}")


In [0]:
def get_pending_files(source_dir: Path) -> list:
    """Returns list of non-hidden files in the pending directory."""
    pending_dir = source_dir / "pending"
    if not pending_dir.exists():
        return []
    return [f for f in pending_dir.iterdir() if f.is_file() and not f.name.startswith('.')]

In [0]:
def main():
    os.sync()
    
    base_837_dir = ROOT_DIR / "source/837"
    pending_837_files = get_pending_files(base_837_dir)
    
    print(f"DEBUG: base_837_dir = {base_837_dir}")
    print(f"DEBUG: pending_837_files = {pending_837_files}")

    # 1. Process 837 Pending Claims Files
    claims_dfs = []
    processed_837_trackers = []

    for file_path in pending_837_files:
        try:
            df, tracked_file = process_edi_claims_file(file_path, base_837_dir)
            claims_dfs.append(df)
            processed_837_trackers.append((tracked_file, base_837_dir))
        except Exception as e:
            print(f"Skipping failed 837 file: {file_path.name}")

    claims_df = pd.concat(claims_dfs, ignore_index=True) if claims_dfs else pd.DataFrame()
    claims_df = normalize_claims_dataframe(claims_df)

    # 2. Read Member Gold Dimension and Measure Library directly from Catalog
    print("Loading Gold Member and Measure Library tables...")
    member_df = spark.table("claimspan.gold.gold_dimmember").toPandas()
    lookup_df = spark.table("claimspan.bronze.measure_library").toPandas()

    # 3. Execute Care Gap Calculation
    try:
        print("\n=== Calculating care gaps ===")
        final_report_df = calculate_care_gaps(
            member_df=member_df,
            claims_df=claims_df,
            lookup_df=lookup_df,
            measurement_year=2026
        )

        print(f"\nExecution Complete! Evaluated {len(final_report_df)} record(s).")

        if not final_report_df.empty:
            display(final_report_df)
            output_file = "/Workspace/Repos/logi@openhealthagents.org/claimspan/ClaimsProcessing/ProcessedCSV/care_gap_report.csv"
            final_report_df.to_csv(output_file, index=False)
            print(f"Saved care gap report to: {output_file}")
        else:
            print("No care gaps evaluated. Check demographic filters or procedure code matches.")

    except Exception as e:
        print(f"Care gap processing failed: {e}")
    
    # 4. Archive processed 837 files
    print("\n=== Archiving processed files ===")
    for tracked_file, base_dir in processed_837_trackers:
        finalize_file_tracking(tracked_file, base_dir, success=True)

In [0]:
if __name__ == "__main__":
    main()

In [0]:
%sql
DESCRIBE TABLE claimspan.bronze.measure_library;

In [0]:
%sql
DESCRIBE TABLE claimspan.gold.gold_dimmember;